# Chatting with your documents

In [1]:
# Check python version: Should be 3.10
!python --version

Python 3.11.9


In [1]:
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", "langchain-openai", 
           "langchain-text-splitters", "pypdf", "pinecone"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")

# langchain version: 1.3.6
# langchain-community version: 0.4.2
# langchain-openai version: 1.2.2
# langchain-text-splitters version: 1.1.2
# pypdf version: 6.10.2
# pinecone version: 7.3.0


langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-openai version: 1.2.2
langchain-text-splitters version: 1.1.2
pypdf version: 6.10.2
pinecone version: 7.3.0


In [14]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads variables from .env

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")

EMBEDDING_DIM=1536
PINECONE_INDEX = 'loxford-quantum-mechanics'

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from pinecone import Pinecone, ServerlessSpec
# from langchain.chains.combine_documents import create_stuff_documents_chain

# Retrieve top 2 results similar to query from VectorDB

In [15]:
## Vector Search DB In Pinecone
pc = Pinecone( api_key=PINECONE_API_KEY )
index = pc.Index(PINECONE_INDEX)


def retrieve_query_results_from_vectorDB(query, k=2):
    matching_results = []

    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small", # cheapest model
        api_key=OPENAI_API_KEY
    )
    query_embedding = embeddings.embed_query(query)

    query_results = index.query(
        vector=query_embedding,
        top_k=k,
        include_metadata=True
    )

    for match in query_results['matches']:
        page_content = match['metadata']['text']
        metadata = {
        "source": match['metadata'].get('source'),
        "page": match['metadata'].get('page'),
        "chunk_id": match['metadata'].get('chunk_id')
        }

        matching_results.append({
            "page_content": page_content,
            "metadata": metadata
        })

    return matching_results


In [16]:
query = "Who are the founders of Loxford Academy?"
matching_results = retrieve_query_results_from_vectorDB(query)

for m in matching_results:
    print(m)
    print("------")

{'page_content': 'Loxford Academy: \nLoxford Academy was founded by Alberto Graciaso Negurasa and Rohit Salvadore Lomax. \nIts address is 5432, Deru Valley on planet Mars.  Loxford Academy has 34,000 employees \nand has a revenue of 99 million dollars. Its employee workforce consists of citizen from  \nAustralia, India, USA, Europe, and many other countries. Its headquarter is in Honda Valley \nlocated on Mars.', 'metadata': {'source': 'Quantum Mechanics-LoxfordAcademy-small.pdf', 'page': 1.0, 'chunk_id': 3.0}}
------
{'page_content': 'concise introduction: we’ll define quantum mechanics, outline its history with a timeline, \nmeet two pioneers (Max Planck and Erwin Schrödinger) via short bios and a comparison \ntable, then unpack essential quantum concepts (wave–particle duality, superposition, \nuncertainty, quantization, quantum states/operators, measurement, entanglement, spin \nand Pauli exclusion, tunneling) with simple analogies. We’ll then glance at modern \napplications (quant

## Create a prompt template

In [30]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

# Create Chain

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Give instructions in prompt so it does NOT hallucinate
prompt = ChatPromptTemplate.from_template(
"""
You are a helpful assistant.

Use ONLY the information provided in the context.

If the answer cannot be found in the context,
reply exactly:

"I don't know."

Context:
{context}

Question:
{user_query}
"""
)

In [33]:
# chain = ( prompt | llm | StrOutputParser())

def generate_answer_from_LLM(query):
    matching_results = retrieve_query_results_from_vectorDB(query)

    # Convert
    matching_docs = [
        Document(page_content=doc['page_content'])
        for doc in matching_results
    ]
    context = "\n\n".join( doc.page_content for doc in matching_docs)
    
    formatted_prompt = prompt.invoke({
        "context": context,
        "user_query": query
    })
    # print("formatted_prompt:", formatted_prompt) # The prompt that goes into LLM
    response = llm.invoke(formatted_prompt)
    answer = response.content

    # 🔹 Keep metadata separately (for you)
    sources = []
    for doc in matching_results:
        sources.append({
            "source": doc['metadata'].get("source"),
            "page":   doc['metadata'].get("page"),
            "chunk":  doc['metadata'].get("chunk_id")
        })
        
    return answer, sources

In [34]:
# Example query
query = "Who are the founders of Loxford Academy ?"
answer, sources = generate_answer_from_LLM(query)
print("Answer:\n", answer)

print("\nSources:")
for s in sources:
    print(s)

Answer:
 The founders of Loxford Academy are Alberto Graciaso Negurasa and Rohit Salvadore Lomax.

Sources:
{'source': 'Quantum Mechanics-LoxfordAcademy-small.pdf', 'page': 1.0, 'chunk': 3.0}
{'source': 'Quantum Mechanics-LoxfordAcademy-small.pdf', 'page': 0.0, 'chunk': 1.0}
